# 01_mini_ablation_binary_crossdomain_10k.ipynb

Edit the CONFIG block, then Run All on Kaggle. This notebook does not perform face detection.

In [ ]:
# CONFIG: mount 00c bundle, raw images, MiniFASNet source and official checkpoint.
RESOURCE_ROOT='/kaggle/input/.../minifasnet_binary_crossdomain_resources_v2'
CELEBA_ROOT='/kaggle/input/.../CelebA_Spoof'
LCC_ROOT='/kaggle/input/.../LCC_FASD_evaluation'
MINIFASNET_SOURCE_PATH='/kaggle/input/.../MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/.../2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/minifasnet_binary_crossdomain_mini_ablation_v2'
RUN_FILTER=None # Example: ['C_copy_pretrained','E_copy_pretrained_csmr']; None runs all six.
RESUME=True; NUM_WORKERS=4; AMP=True
DATA_SEED=43; TRAIN_SEED=100
LEGACY_BATCH_SIZE=128; COPY_BATCH_SIZE=256
LEGACY_MAX_EPOCHS=12; LEGACY_EARLIEST_STOP=5; LEGACY_PATIENCE=3
COPY_MAX_EPOCHS=20; COPY_EARLIEST_STOP=8; COPY_PATIENCE=4
CSMR_WARMUP_EPOCHS=1; CSMR_LAMBDA=.2; CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10; CSMR_GAIN_RANGE=[.65,.90]; CSMR_PRESERVE_DC=True; SELECTION_CHUNK_SIZE=32
CROP_SCALE=2.7; INPUT_SIZE=80; METRIC_TIE_EPS=1e-12
EXPECTED_CONTRACT_SHA='2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91'
EXPECTED_PARENT_SHA='aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
EXPECTED_DETECTOR_SHA='5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
EXPECTED_MODEL_SHA='e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7'
EXPECTED_PRETRAINED_SHA='a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0'

import copy, csv, hashlib, importlib.util, inspect, json, math, os, random, time
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def hash_json(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def ordered_sha(keys): return hashlib.sha256('\n'.join(map(str,keys)).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
def write_json(path,x): Path(path).write_text(json.dumps(x,indent=2,sort_keys=True,allow_nan=False)+'\n')
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def save_torch(path,value):
    tmp=Path(str(path)+'.tmp');torch.save(value,tmp);os.replace(tmp,path)
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if random.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=random.uniform(.95,1.05)
        cx+=random.uniform(-.05,.05)*base;cy+=random.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))
def mild_aug(patch):
    if random.random()<.5:patch=cv2.flip(patch,1)
    if random.random()<.3:
        alpha=random.uniform(.9,1.1);beta=random.uniform(-.1,.1)*255
        patch=np.clip(patch.astype(np.float32)*alpha+beta,0,255).astype(np.uint8)
    return patch
# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

RUNS=OrderedDict([
 ('A_legacy_pretrained',{'init':'pretrained','copied':False,'csmr':False}),
 ('B_legacy_scratch',{'init':'scratch','copied':False,'csmr':False}),
 ('C_copy_pretrained',{'init':'pretrained','copied':True,'csmr':False}),
 ('D_copy_scratch',{'init':'scratch','copied':True,'csmr':False}),
 ('E_copy_pretrained_csmr',{'init':'pretrained','copied':True,'csmr':True}),
 ('F_copy_scratch_csmr',{'init':'scratch','copied':True,'csmr':True})])
if RUN_FILTER is not None and (len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):raise ValueError('RUN_FILTER must contain unique frozen run IDs')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for p in [root/'config.json',root/'provenance.json',root/'manifest_fingerprints.json',root/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md',
  root/'labels/celeba_auxiliary_labels.csv',root/'bbox/celeba_bbox_cache_full.json',root/'bbox/lcc_official_evaluation_bbox_cache.json',
  root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',root/'manifests/crossdomain_celeba_source_dev_full.csv',
  root/'manifests/lcc_official_evaluation_full.csv',source,pretrained]:
    if not p.is_file():raise FileNotFoundError(p)
if sha(root/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md')!=EXPECTED_CONTRACT_SHA:raise RuntimeError('V2 contract mismatch')
if sha(source)!=EXPECTED_MODEL_SHA or sha(pretrained)!=EXPECTED_PRETRAINED_SHA:raise RuntimeError('Model source/pretrained hash mismatch')
config=read_json(root/'config.json');prov=read_json(root/'provenance.json');fps=read_json(root/'manifest_fingerprints.json')
if config['v2_contract_sha256']!=EXPECTED_CONTRACT_SHA or prov['v2_contract_sha256']!=EXPECTED_CONTRACT_SHA or prov['parent_contract_sha256']!=EXPECTED_PARENT_SHA:raise RuntimeError('Resource contract mismatch')
if prov['detector_sha256']!=EXPECTED_DETECTOR_SHA or prov['scrfd_execution_provider']!='CUDAExecutionProvider' or prov['mini_data_seed']!=DATA_SEED:raise RuntimeError('Resource provenance mismatch')
if config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:raise RuntimeError('Input contract mismatch')
for split in ('train','test'):
    metadata=Path(CELEBA_ROOT)/'metas/intra_test'/(split+'_label.json')
    if not metadata.is_file() or sha(metadata)!=prov['source_hashes'][split+'_label_json']:raise RuntimeError('Raw CelebA metadata hash mismatch: '+split)
if (Path(LCC_ROOT)/'LCC_FASD_evaluation').is_dir():LCC_ROOT=str(Path(LCC_ROOT)/'LCC_FASD_evaluation')
for kind in ('celeba','lcc'):
    if sha(root/'bbox'/(kind+'_bbox_cache_full.json' if kind=='celeba' else 'lcc_official_evaluation_bbox_cache.json'))!=prov[kind+'_bbox_sha256']:
        # CelebA file has a distinct historical name.
        if kind!='celeba' or sha(root/'bbox/celeba_bbox_cache_full.json')!=prov['celeba_bbox_sha256']:raise RuntimeError('Cache SHA mismatch')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records'];lcc_records=read_json(root/'bbox/lcc_official_evaluation_bbox_cache.json')['records']
for cache in (records,lcc_records):
    for key,rec in cache.items():
        if rec.get('usable_for_pad'):require_record(rec,key)
def load_frame(name,key='sample_key'):
    p=root/'manifests'/(name+'.csv');f=pd.read_csv(p,keep_default_na=False);fp=fps[name]
    if sha(p)!=fp['file_sha256'] or len(f)!=fp.get('sample_count',fp.get('candidate_n')):raise RuntimeError('Manifest SHA/count mismatch: '+name)
    if key=='sample_key' and ordered_sha(f[key])!=fp['ordered_sample_key_sha256']:raise RuntimeError('Manifest order mismatch: '+name)
    if key=='path' and ordered_sha(f[key])!=fp['ordered_path_sha256']:raise RuntimeError('LCC manifest order mismatch')
    if f[key].duplicated().any():raise RuntimeError('Duplicate manifest key: '+name)
    return f
train=load_frame('mini_train10k');val=load_frame('mini_val3k');test=load_frame('crossdomain_celeba_source_dev_full');lcc=load_frame('lcc_official_evaluation_full','path')
if len(train)!=10000 or len(val)!=3000 or len(test)!=67170 or len(lcc)!=7580 or int((lcc.label==0).sum())!=314 or int((lcc.label==1).sum())!=7266:raise RuntimeError('Population invariant')
if set(train.sample_key)&set(val.sample_key) or set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Mini leakage')
for frame in (train,val,test):
    for row in frame.itertuples(index=False):
        rec=records[str(row.sample_key)]
        if str(row.localization_status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']) or str(row.bbox_origin)!=rec['bbox_origin']:raise RuntimeError('Manifest/cache mismatch')
        if rec['usable_for_pad']:require_record(rec,str(row.sample_key))
for row in lcc.itertuples(index=False):
    rec=lcc_records[str(row.path)]
    if str(row.status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']):raise RuntimeError('LCC cache mismatch')
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
if sha(root/'labels/celeba_auxiliary_labels.csv')!=prov['auxiliary_labels_sha256']:raise RuntimeError('Aux SHA mismatch')
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing auxiliary labels')
for frame in (train,val):
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:raise RuntimeError('Mini data invariant')
if not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required; no CPU fallback for this experiment')
print('GPU',torch.cuda.get_device_name(0),'PyTorch',torch.__version__,'CUDA',torch.version.cuda,'Albumentations',A.__version__,'OpenCV',cv2.__version__)
print('Resource/source hashes:',prov['source_hashes'],'manifest hashes:',{k:v['file_sha256'] for k,v in fps.items()},'model',sha(source),'pretrained',sha(pretrained))
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source);module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key
class LCCData(Dataset):
    def __init__(self,frame):self.rows=frame.loc[frame.usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[key],key)
        image=cv2.imread(str(Path(LCC_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}
def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[]
    with torch.no_grad():
        for batch in dl:
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def radial_masks(size=INPUT_SIZE,device=device):
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(CSMR_BAND_CENTERS,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*CSMR_BAND_SIGMA**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)
def select_worst_view(model,base,binary,masks):
    was_training=model.training;model.eval();chosen=[];bands=[];gains_out=[];clips_out=[]
    try:
        with torch.no_grad():
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=torch.empty(len(x),device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):logits=model(views[:,band])
                    margins.append(aligned_margin(logits,y))
                ix=torch.stack(margins,dim=1).argmin(dim=1);row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach());gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(z) for z in (chosen,bands,gains_out,clips_out))
def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()}
def restore_rng(r):
    random.setstate(r['python']);np.random.set_state(r['numpy']);torch.set_rng_state(r['torch']);torch.cuda.set_rng_state_all(r['cuda'])
def run_config(name,spec):
    copied=spec['copied']
    return {'run_id':name,'spec':spec,'train_seed':TRAIN_SEED,'data_seed':DATA_SEED,'contract_sha256':EXPECTED_CONTRACT_SHA,
      'train_manifest_sha256':fps['mini_train10k']['file_sha256'],'val_manifest_sha256':fps['mini_val3k']['file_sha256'],
      'test_manifest_sha256':fps['crossdomain_celeba_source_dev_full']['file_sha256'],'lcc_manifest_sha256':fps['lcc_official_evaluation_full']['file_sha256'],
      'celeba_bbox_sha256':prov['celeba_bbox_sha256'],'lcc_bbox_sha256':prov['lcc_bbox_sha256'],
      'model_source_sha256':sha(source),'pretrained_checkpoint_sha256':sha(pretrained),
      'batch_size':COPY_BATCH_SIZE if copied else LEGACY_BATCH_SIZE,'max_epochs':COPY_MAX_EPOCHS if copied else LEGACY_MAX_EPOCHS,
      'earliest_early_stop_epoch':COPY_EARLIEST_STOP if copied else LEGACY_EARLIEST_STOP,
      'patience':COPY_PATIENCE if copied else LEGACY_PATIENCE,'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4} if copied else
        {'name':'AdamW','backbone_lr':2e-4 if spec['init']=='pretrained' else 1e-3,'classifier_lr':1e-3,'weight_decay':1e-4},
      'scheduler':{'type':'MultiStepLR','milestones':[6,14],'gamma':.2,'step':'after each completed epoch'} if copied else None,
      'label_smoothing':0 if copied else .1,'grad_clip':5.0 if not copied else None,
      'augmentation':'literature adapted RGB-only augmentation then BGR MiniFAS input' if copied else 'historical mild',
      'albumentations_version':A.__version__,'augmentation_api':AUGMENTATION_API if copied else None,
      'bbox_jitter':'20pct scale .95-1.05 shift +/- .05','input':'80x80 BGR float [0,1] 2.7x',
      'csmr':{'warmup_epochs':1,'lambda':CSMR_LAMBDA,'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,'gain':CSMR_GAIN_RANGE,'preserve_dc':True,'chunk_size':SELECTION_CHUNK_SIZE} if spec['csmr'] else None}

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True)
    cfg=run_config(name,spec);cfg_hash=hash_json(cfg);cfg_path=folder/'config.json';last_path=folder/'last_state.pth';best_path=folder/'best.pth'
    if cfg_path.exists():
        if read_json(cfg_path)!=cfg:raise RuntimeError(name+' config hash changed; use a new output directory')
    else:write_json(cfg_path,cfg)
    model,audit=init_model(spec)
    audit_path=folder/'initialization_audit.json'
    if audit_path.exists() and read_json(audit_path)!=audit:raise RuntimeError('Initialization audit drift')
    write_json(audit_path,audit);model=model.to(device)
    complete_path=folder/'completion.json'
    if complete_path.exists():
        done=read_json(complete_path)
        required=[folder/(kind+'_'+suffix) for kind in ('val','test','lcc') for suffix in ('metrics.json','predictions.csv')]
        if done['config_hash']!=cfg_hash or not best_path.is_file() or done['best_checkpoint_sha256']!=sha(best_path) or not all(p.is_file() for p in required):raise RuntimeError('Completed run provenance/output mismatch')
        print(name,'already completed and verified; no held-out re-evaluation')
        return done
    if best_path.exists() and not last_path.exists():raise RuntimeError('Orphan best checkpoint without resume state')
    if spec['copied']:
        optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
        scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=[6,14],gamma=.2)
    elif spec['init']=='pretrained':
        optimizer=torch.optim.AdamW([{'params':model.trunk.prob.parameters(),'lr':1e-3},
          {'params':(p for n,p in model.trunk.named_parameters() if not n.startswith('prob.')),'lr':2e-4}],weight_decay=1e-4);scheduler=None
    else:optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4);scheduler=None
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    history=[];diag=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1;warmup_meta=None
    if last_path.exists():
        if not RESUME:raise RuntimeError(name+' has incomplete state; enable RESUME or new directory')
        s=torch.load(last_path,map_location='cpu',weights_only=False)
        if s['config_hash']!=cfg_hash or s['initial_state_sha256']!=audit['initial_state_sha256'] or s['run_id']!=name:raise RuntimeError('Resume configuration/init mismatch')
        if s['best_epoch']:
            if not best_path.is_file() or s['best_checkpoint_sha256']!=sha(best_path):raise RuntimeError('Resume best checkpoint mismatch')
        model.load_state_dict(s['model_state']);optimizer.load_state_dict(s['optimizer_state'])
        if scheduler:scheduler.load_state_dict(s['scheduler_state'])
        scaler.load_state_dict(s['scaler_state']);history=s['history'];diag=s['csmr_diagnostics'];best_acer=s['best_acer'];best_auc=s['best_auc'];best_epoch=s['best_epoch'];no_improve=s['no_improve_count'];warmup_meta=s['warmup_metadata'];start=s['epoch']+1
        restore_rng(s['rng']);print(name,'resume from epoch',start)
    batch=cfg['batch_size'];train_ds=CelebaData(train,True,spec['copied']);val_dl=loader(CelebaData(val),batch)
    masks=radial_masks() if spec['csmr'] else None
    max_epochs=cfg['max_epochs'];best_exists=best_path.is_file()
    if start<=max_epochs and not (history and history[-1].get('stop_reached')):
        for epoch in range(start,max_epochs+1):
            model.train();totals=Counter();nseen=0;bands_count=Counter();active=spec['csmr'] and epoch>CSMR_WARMUP_EPOCHS
            for x,y3,binary,spoof,lighting,attrs,_ in loader(train_ds,batch,True,epoch):
                x=x.to(device,non_blocking=True);y3=y3.to(device);binary=binary.to(device);spoof=spoof.to(device);lighting=lighting.to(device);attrs=attrs.to(device)
                optimizer.zero_grad(set_to_none=True)
                if active:
                    selected,band,gain,clip=select_worst_view(model,x,binary,masks)
                    bands_count.update(band.cpu().tolist())
                with torch.cuda.amp.autocast(enabled=use_amp):
                    if spec['copied']:
                        clean=model(x,aux=True);pad=F.cross_entropy(clean[0],binary);ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs)
                        if active:
                            worst=model(selected);ce_worst=F.cross_entropy(worst,binary)
                            mc=aligned_margin(clean[0],binary);mw=aligned_margin(worst,binary);harmful=mw<mc.detach()
                            margin=(F.smooth_l1_loss(mw,mc.detach(),reduction='none')*harmful.float()).mean()
                            classification=.5*pad+.5*ce_worst
                            loss=classification+CSMR_LAMBDA*margin+.1*ls+.1*ll+la
                        else:
                            ce_worst=pad*0;margin=pad*0;classification=pad;harmful=torch.zeros_like(binary,dtype=torch.bool);mc=mw=pad.detach().expand_as(binary)
                            loss=pad+.1*ls+.1*ll+la
                    else:
                        logits=model(x);pad=F.cross_entropy(logits,y3,label_smoothing=.1);loss=pad
                        ls=ll=la=ce_worst=margin=pad*0;classification=pad;harmful=torch.zeros_like(y3,dtype=torch.bool);mc=mw=pad.detach().expand_as(y3)
                scaler.scale(loss).backward()
                if not spec['copied']:scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
                try:scaler.step(optimizer);scaler.update()
                except torch.cuda.OutOfMemoryError as exc:raise RuntimeError('Batch 256 OOM. Stop; restart C/D/E/F consistently with an explicit new protocol batch size.') from exc
                count=len(x);nseen+=count
                for key,value in {'train_loss':loss,'pad_loss':pad,'spoof_loss':ls,'lighting_loss':ll,'attr_loss':la,'ce_clean':pad,'ce_worst':ce_worst,
                   'classification_loss':classification,'margin_loss':margin,'harmful_fraction':harmful.float().mean(),'clean_margin_mean':mc.float().mean(),
                   'worst_margin_mean':mw.float().mean()}.items():totals[key]+=float(value.detach())*count
            val_keys,val_y,val_logits=infer(model,val_dl)
            if val_keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order mismatch')
            val_scores=pad_score(val_logits);threshold=threshold_from_val(val_y,val_scores);vm=metrics(val_y,val_scores,threshold,len(val))
            eligible=not spec['csmr'] or active
            improved=eligible and (vm['acer']<best_acer-METRIC_TIE_EPS or (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if eligible:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    save_torch(best_path,{'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},'epoch':epoch,'config_hash':cfg_hash,'val_acer':best_acer,'val_auc':best_auc})
                    best_exists=True
                else:no_improve+=1
            if spec['csmr'] and not active:
                save_torch(folder/'warmup.pth',{'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},'epoch':epoch,'config_hash':cfg_hash})
                warmup_meta={'epoch':epoch,'val_metrics':vm,'checkpoint_sha256':sha(folder/'warmup.pth')};write_json(folder/'warmup_metrics.json',warmup_meta)
            lr=optimizer.param_groups[0]['lr'];row={'epoch':epoch,'lr':lr,'phase':'csmr' if active else 'warmup' if spec['csmr'] else 'train',
               'eligible_for_final_checkpoint':eligible,'val_acer':vm['acer'],'val_auc':vm['auc'],'val_eer':vm['eer'],'threshold':threshold,
               'best_so_far':improved,'no_improve_count':no_improve,**{k:v/nseen for k,v in totals.items()}}
            if spec['csmr']:
                row['selected_band_frequencies']={str(i):bands_count[i] for i in range(3)}
                diag.append({'epoch':epoch,'phase':row['phase'],**{k:row[k] for k in ('ce_clean','ce_worst','classification_loss','margin_loss','harmful_fraction','clean_margin_mean','worst_margin_mean','selected_band_frequencies')}})
            stop=epoch==max_epochs or (eligible and epoch>=cfg['earliest_early_stop_epoch'] and no_improve>=cfg['patience'])
            row['stop_reached']=stop;history.append(row)
            if scheduler:scheduler.step() # after epoch: milestones 6/14 affect following epoch 7/15 in current PyTorch.
            state={'run_id':name,'config_hash':cfg_hash,'initial_state_sha256':audit['initial_state_sha256'],'epoch':epoch,
              'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
              'best_state':torch.load(best_path,map_location='cpu',weights_only=False)['state_dict'] if best_epoch else None,
              'best_checkpoint_sha256':sha(best_path) if best_epoch else None,'optimizer_state':optimizer.state_dict(),
              'scheduler_state':scheduler.state_dict() if scheduler else None,'scaler_state':scaler.state_dict(),'rng':rng_state(),
              'history':history,'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,'no_improve_count':no_improve,
              'best_csmr_epoch':best_epoch if spec['csmr'] else None,'best_csmr_acer':best_acer if spec['csmr'] else None,
              'best_csmr_auc':best_auc if spec['csmr'] else None,'csmr_no_improve_count':no_improve if spec['csmr'] else None,
              'warmup_metadata':warmup_meta,'csmr_diagnostics':diag}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['csmr']:pd.DataFrame(diag).to_csv(folder/'csmr_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'no improve',no_improve,flush=True)
            if stop:break
    if not best_exists or not best_path.is_file():raise RuntimeError('No eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=False)
    if best['config_hash']!=cfg_hash or best['epoch']!=best_epoch or (spec['csmr'] and best_epoch<=CSMR_WARMUP_EPOCHS):raise RuntimeError('Best checkpoint policy mismatch')
    model.load_state_dict(best['state_dict']);model.to(device)
    val_keys,val_y,val_logits=infer(model,val_dl);val_scores=pad_score(val_logits);locked=threshold_from_val(val_y,val_scores)
    val_metrics=metrics(val_y,val_scores,locked,len(val));write_json(folder/'val_metrics.json',val_metrics)
    def prediction_frame(frame,keys,y,z,threshold,kind):
        scored=pd.DataFrame({'key':keys,'three_class_label':y,'pad_logit_score':pad_score(z),'pred_real':pad_score(z)>=threshold})
        for j in range(z.shape[1]):scored[f'logit_{j}']=z[:,j]
        if z.shape[1]==3:scored['pred_class_3']=z.argmax(axis=1)
        if kind=='val':return scored
        base=frame.copy().rename(columns={'sample_key':'key','path':'key'})
        full=base.merge(scored.drop(columns=['three_class_label'],errors='ignore'),on='key',how='left',validate='one_to_one',sort=False)
        if full.loc[base.usable_for_pad.astype(bool),'pad_logit_score'].isna().any() or full.loc[~base.usable_for_pad.astype(bool),'pad_logit_score'].notna().any():raise RuntimeError('Candidate/scored accounting mismatch')
        return full
    prediction_frame(val,val_keys,val_y,val_logits,locked,'val').to_csv(folder/'val_predictions.csv',index=False)
    freeze={'config_hash':cfg_hash,'best_checkpoint_sha256':sha(best_path),'selected_epoch':best_epoch,'locked_threshold':locked}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.exists() and read_json(freeze_path)!=freeze:raise RuntimeError('Held-out evaluation freeze mismatch')
    write_json(freeze_path,freeze)
    # The only held-out inference is after checkpoint and source threshold are locked.
    for kind,frame,ds,keycol in [('test',test.loc[test.usable_for_pad.astype(bool)],CelebaData(test.loc[test.usable_for_pad.astype(bool)]),'sample_key'),
                                  ('lcc',lcc.loc[lcc.usable_for_pad.astype(bool)],LCCData(lcc),'path')]:
        metrics_path=folder/(kind+'_metrics.json');predictions_path=folder/(kind+'_predictions.csv')
        if metrics_path.exists() or predictions_path.exists():
            if not metrics_path.is_file() or not predictions_path.is_file():raise RuntimeError('Partial held-out evaluation output: '+kind)
            print(kind,'already evaluated under the matching freeze; skipping repeated inference')
            continue
        keys,y,z=infer(model,loader(ds,batch),kind=='lcc')
        if keys!=frame[keycol].astype(str).tolist():raise RuntimeError(kind+' scored order mismatch')
        candidates=test if kind=='test' else lcc;mm=metrics(y,pad_score(z),locked,len(candidates));write_json(folder/(kind+'_metrics.json'),mm)
        prediction_frame(candidates,keys,y,z,locked,kind).to_csv(folder/(kind+'_predictions.csv'),index=False)
    if spec['csmr']:
        p=pd.read_csv(folder/'test_predictions.csv');pd.DataFrame([{'three_class_label':int(c),'candidate_n':int((p.three_class_label==c).sum()),
          'scored_n':int(((p.three_class_label==c)&p.pad_logit_score.notna()).sum())} for c in range(3)]).to_csv(folder/'class_diagnostics.csv',index=False)
    result={'run_id':name,'config_hash':cfg_hash,'selected_epoch':best_epoch,'best_checkpoint_sha256':sha(best_path),'locked_threshold':locked,
      'second_lr_decay_reached':bool(history and history[-1]['epoch']>=15) if spec['copied'] else None,
      'val_auc':val_metrics['auc'],'val_eer':val_metrics['eer'],'val_acer':val_metrics['acer'],
      **{f'{kind}_{metric}':read_json(folder/(kind+'_metrics.json'))[metric] for kind in ('test','lcc') for metric in ('auc','eer','acer','detector_coverage')}}
    write_json(complete_path,result)
    return result

experiment={'schema':'binary_crossdomain_mini_ablation_v2','runs':RUNS,'run_filter':RUN_FILTER,'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,
 'v2_contract_sha256':EXPECTED_CONTRACT_SHA,'resource_provenance_sha256':sha(root/'provenance.json'),
 'manifest_fingerprints':fps,'model_source_sha256':sha(source),'pretrained_checkpoint_sha256':sha(pretrained)}
if (out/'experiment_config.json').is_file():
    old=read_json(out/'experiment_config.json')
    if {k:v for k,v in old.items() if k!='run_filter'}!={k:v for k,v in experiment.items() if k!='run_filter'}:raise RuntimeError('Experiment configuration drift')
write_json(out/'experiment_config.json',experiment)
write_json(out/'provenance.json',{'resource':prov,'experiment_config_sha256':hash_json({k:v for k,v in experiment.items() if k!='run_filter'}),
 'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'albumentations':A.__version__,'augmentation_api':AUGMENTATION_API,'opencv':cv2.__version__})
for run_id,spec in RUNS.items():
    if RUN_FILTER is None or run_id in RUN_FILTER:
        try:run_one(run_id,spec)
        except torch.cuda.OutOfMemoryError as exc:raise RuntimeError('OOM at frozen batch size. Stop; do not reduce batch for only one run.') from exc
summary=[]
for run_id in RUNS:
    folder=out/'runs'/run_id
    if all((folder/(kind+'_metrics.json')).is_file() for kind in ('val','test','lcc')):
        if read_json(folder/'config.json')!=run_config(run_id,RUNS[run_id]):raise RuntimeError('Summary run configuration drift: '+run_id)
        row={'run_id':run_id,'init':RUNS[run_id]['init'],'logits':2 if RUNS[run_id]['copied'] else 3,
         'auxiliary':RUNS[run_id]['copied'],'csmr':RUNS[run_id]['csmr']}
        for kind in ('val','test','lcc'):
            m=read_json(folder/(kind+'_metrics.json'))
            for key,value in m.items():row[kind+'_'+key]=value
        summary.append(row)
if summary:
    pd.DataFrame(summary).to_csv(out/'comparison_summary.csv',index=False);write_json(out/'comparison_summary.json',summary)
    report=['# Mini binary cross-domain pilot','',f'Completed runs: {len(summary)}/6.','',
      'C vs E and D vs F isolate CSMR. A/B vs C/D compare recipes with multiple changed factors.',
      'LCC is pilot evidence only and was never used for checkpoint selection or threshold calibration.','',
      'All Test/LCC metrics use each run\'s locked Mini-Val3K threshold.','']
    for row in summary:
        completion=read_json(out/'runs'/row['run_id']/'completion.json')
        report.append(f"- {row['run_id']}: Val ACER {row['val_acer']:.6f}; Test AUC {row['test_auc']:.6f}; LCC AUC {row['lcc_auc']:.6f}; LCC EER {row['lcc_eer']:.6f}; second LR decay reached: {completion['second_lr_decay_reached']}.")
    (out/'mini_ablation_report.md').write_text('\n'.join(report)+'\n')
print('Completed summary rows:',len(summary),'of 6. Outputs:',out)
